# 🧪 PaliGemma Test - Mix 224 Model

**TFG: VisionLeaderUFV**  
**Author:** Alberto Gómez Soteres  
**Universidad Francisco de Vitoria**

---

This notebook tests the **PaliGemma 3B Mix 224** model to verify that:
- All metrics are calculated correctly
- Confusion matrices are generated properly
- Visualizations work as expected

## Model Under Test:
- **Model**: `google/paligemma-3b-mix-224`
- **Description**: PaliGemma 1 - 3B fine-tuned mix - 224px

## What This Notebook Does:
1. Loads and evaluates the model on UFV Campus dataset
2. Calculates all metrics (Exact Match, Keyword Recall, BERTScore, Throughput)
3. Generates confusion matrices (YES/NO, OPEN/CLOSED, UP/DOWN, Elements)
4. Displays visual confusion matrices with matplotlib/seaborn
5. Shows example predictions for verification

## 1️⃣ Setup and Installation

In [ ]:
# Install required packages
!pip install -q transformers>=4.40.0 accelerate bitsandbytes
!pip install -q torch torchvision
!pip install -q pandas numpy tqdm Pillow matplotlib seaborn

!pip install -q bert_score  # For BERTScore metric

print("✅ All packages installed successfully!")

In [ ]:
# Mount Google Drive (for data only)
from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive mounted (for data access)!")

In [ ]:
# Setup project - Clone from GitHub (Private Repo)
import os
import sys
import subprocess

# ============================================================
# CONFIGURATION - PRIVATE REPO
# ============================================================
GITHUB_USER = "AlbertoGomezS"
GITHUB_REPO = "tfg-AlbertoGomez"
LOCAL_PROJECT_PATH = '/content/tfg'

# GitHub Personal Access Token (create at: https://github.com/settings/tokens)
# Required scopes: repo (full control of private repositories)
GITHUB_TOKEN = "YOUR_GITHUB_TOKEN"  # <-- REPLACE THIS

# ============================================================

# Clone or update from GitHub
if os.path.exists(LOCAL_PROJECT_PATH):
    # Update existing repo
    os.chdir(LOCAL_PROJECT_PATH)
    result = subprocess.run(['git', 'pull'], capture_output=True, text=True)
    print(result.stdout)
    if result.returncode == 0:
        print("✅ Repository updated from GitHub")
    else:
        print(f"⚠️ Git pull issue: {result.stderr}")
else:
    # Clone with authentication (private repo)
    clone_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{GITHUB_REPO}.git"
    result = subprocess.run(['git', 'clone', clone_url, LOCAL_PROJECT_PATH], capture_output=True, text=True)
    
    if result.returncode == 0:
        os.chdir(LOCAL_PROJECT_PATH)
        print(f"✅ Repository cloned from GitHub (private)")
    else:
        print(f"❌ Clone failed. Check your GITHUB_TOKEN")
        print(f"   Error: {result.stderr}")
        raise Exception("Could not clone repository - check token")

# Add src to path
sys.path.insert(0, LOCAL_PROJECT_PATH)

print(f"📁 Working directory: {os.getcwd()}")

In [ ]:
# Import project modules
from src.model_loader import load_paligemma, unload_model, print_gpu_memory, get_model_info
from src.dataset_loader import (
    load_questions_config,
    load_dataset,
    create_evaluation_items,
    print_dataset_summary,
    load_per_image_answers
)
from src.inference import (
    run_inference_on_dataset,
    print_inference_summary,
    results_to_dict
)
from src.metrics import (
    calculate_all_metrics,
    print_metrics_summary,
    print_confusion_matrices,
    get_gpu_memory_usage,
    reset_gpu_memory_stats
)

import json
import pandas as pd
from datetime import datetime

print("✅ All modules imported successfully!")

## 2️⃣ Configuration

In [ ]:
# ============================================================
# CONFIGURATION - PATHS
# ============================================================

# Data is stored in Google Drive (too large for Git)
DATA_PATH = '/content/drive/MyDrive/tfg/data/UFV Campus'  # Only UFV Campus dataset

# Config and src are cloned from GitHub
CONFIG_PATH = '/content/tfg/config'

# Results stored in Colab /content directory (local)
RESULTS_PATH = '/content/results/test_mix_224'

# Create results directory
os.makedirs(RESULTS_PATH, exist_ok=True)

# Verify data path exists
if not os.path.exists(DATA_PATH):
    print(f"⚠️ Data path not found: {DATA_PATH}")
    print("   Please upload your UFV Campus data folder to Google Drive")
else:
    print(f"✅ Data path found: {DATA_PATH}")
    
print(f"📁 Config path: {CONFIG_PATH}")
print(f"📁 Results path: {RESULTS_PATH}")

# Model to test
MODEL_TO_TEST = "google/paligemma-3b-mix-224"
print(f"\n🧪 Model to test: {MODEL_TO_TEST}")

In [ ]:
# Load configurations
questions_config = load_questions_config(f"{CONFIG_PATH}/questions.json")

# Load per-image expected answers
per_image_answers = load_per_image_answers(f"{CONFIG_PATH}/ufv_expected_answers.json")

print(f"✅ Loaded {len(questions_config['categories'])} question categories")
print(f"✅ Loaded expected answers for {len(per_image_answers)} images")

In [ ]:
# Load UFV Campus dataset
samples = load_dataset(DATA_PATH, questions_config, load_images=False)

# Print dataset summary
print_dataset_summary(samples)

# Create evaluation items (image + question pairs)
eval_items = create_evaluation_items(samples, questions_config, per_image_answers=per_image_answers)

print(f"\n📊 Total evaluation items: {len(eval_items)}")
print(f"   (Each image has multiple questions depending on category)")

## 3️⃣ HuggingFace Login

In [ ]:
# Login to HuggingFace (required for PaliGemma access)
from huggingface_hub import login

# IMPORTANT: Replace with your HuggingFace token
# Get your token from: https://huggingface.co/settings/tokens
# Make sure you have accepted the PaliGemma license

HF_TOKEN = "YOUR_HUGGINGFACE_TOKEN"  # <-- REPLACE THIS

if HF_TOKEN != "YOUR_HUGGINGFACE_TOKEN":
    login(token=HF_TOKEN)
    print("✅ Logged in to HuggingFace")
else:
    print("⚠️ Please set your HuggingFace token above")
    print("   1. Get your token from: https://huggingface.co/settings/tokens")
    print("   2. Accept the PaliGemma license")
    print("   3. Replace 'YOUR_HUGGINGFACE_TOKEN' with your actual token")

## 4️⃣ Run Evaluation

In [ ]:
print(f"🚀 Starting evaluation of model: {MODEL_TO_TEST}")
print(f"📊 Total items to evaluate: {len(eval_items)}")
print("=" * 60)

# Get model info
model_info = get_model_info(MODEL_TO_TEST)
print(f"📦 Model: {MODEL_TO_TEST}")
print(f"   Version: {model_info['version']}, Size: {model_info['size']}, "
      f"Type: {model_info['type']}, Resolution: {model_info['resolution']}")

# Reset GPU memory stats
reset_gpu_memory_stats()

# Load model
print("\n🔄 Loading model...")
try:
    model, processor = load_paligemma(MODEL_TO_TEST)
except Exception as e:
    print(f"❌ Error loading model: {e}")
    raise

# Run inference
print("\n🔬 Running inference...")
results = run_inference_on_dataset(
    model,
    processor,
    eval_items,
    max_new_tokens=100,
    verbose=True  # Show detailed output
)

# Convert to dict for metrics
results_dict = results_to_dict(results)

# Calculate metrics
vram_usage = get_gpu_memory_usage()
metrics = calculate_all_metrics(results_dict, vram_usage=vram_usage)

# Print summary
print_metrics_summary(metrics)

# Save results
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
model_name_safe = MODEL_TO_TEST.replace("/", "_").replace("-", "_")

# Save raw results
results_file = f"{RESULTS_PATH}/{model_name_safe}_results_{timestamp}.json"
with open(results_file, 'w') as f:
    json.dump(results_dict, f, indent=2)
print(f"\n💾 Results saved to: {results_file}")

# Save metrics
metrics_file = f"{RESULTS_PATH}/{model_name_safe}_metrics_{timestamp}.json"
with open(metrics_file, 'w') as f:
    json.dump(metrics, f, indent=2)
print(f"💾 Metrics saved to: {metrics_file}")

In [ ]:
# Display confusion matrices with visualizations
print_confusion_matrices(metrics, show_plots=True)

## 6️⃣ Verify Calculations - Sample Predictions

In [ ]:
# Show some example predictions to verify calculations
print("\n" + "=" * 60)
print("SAMPLE PREDICTIONS FOR VERIFICATION")
print("=" * 60)

# Get some examples of each question type
closed_examples = [r for r in results_dict if r.get('question_type') == 'closed'][:5]
element_examples = [r for r in results_dict if r.get('question_type') == 'element_list'][:5]
caption_examples = [r for r in results_dict if r.get('question_type') == 'caption'][:3]

print("\n📋 Closed Questions (YES/NO, OPEN/CLOSED, UP/DOWN):")
print("-" * 60)
for i, ex in enumerate(closed_examples, 1):
    print(f"\n{i}. Image: {ex.get('image_id', 'N/A')}")
    print(f"   Category: {ex.get('category', 'N/A')}")
    print(f"   Question: {ex.get('question_text', 'N/A')}")
    print(f"   Model Response: {ex.get('model_response', 'N/A')}")
    print(f"   Expected Answer: {ex.get('expected_answer', 'N/A')}")
    # Check if match
    pred = ex.get('model_response', '').lower().strip()
    expected = ex.get('expected_answer', '').lower().strip()
    match = "✅" if expected in pred or pred in expected else "❌"
    print(f"   Match: {match}")

print("\n📋 Element Identification Questions:")
print("-" * 60)
for i, ex in enumerate(element_examples, 1):
    print(f"\n{i}. Image: {ex.get('image_id', 'N/A')}")
    print(f"   Category: {ex.get('category', 'N/A')}")
    print(f"   Question: {ex.get('question_text', 'N/A')}")
    print(f"   Model Response: {ex.get('model_response', 'N/A')}")
    expected_keywords = ex.get('expected_keywords_primary', [])
    print(f"   Expected Keywords (Primary): {expected_keywords}")
    # Check if any keyword matches
    pred = ex.get('model_response', '').lower()
    match = "✅" if any(kw.lower() in pred for kw in expected_keywords) else "❌"
    print(f"   Match: {match}")

print("\n📋 Caption Questions:")
print("-" * 60)
for i, ex in enumerate(caption_examples, 1):
    print(f"\n{i}. Image: {ex.get('image_id', 'N/A')}")
    print(f"   Category: {ex.get('category', 'N/A')}")
    print(f"   Model Response: {ex.get('model_response', 'N/A')[:100]}...")
    print(f"   Reference Answer: {ex.get('reference_answer', 'N/A')}")

print("\n" + "=" * 60)

In [ ]:
# Final summary
overall = metrics.get('overall', {})
print("\n" + "=" * 60)
print("FINAL SUMMARY")
print("=" * 60)
print(f"\n✅ Model evaluated: {MODEL_TO_TEST}")
print(f"✅ Total evaluation items: {len(results_dict)}")
print(f"✅ Metrics calculated successfully")
print(f"✅ Confusion matrices generated")

print(f"\n📊 Key Metrics:")
print(f"  Exact Match:        {overall.get('exact_match', 0):.4f}")
print(f"  Keyword Recall:     {overall.get('keyword_recall', 0):.4f}")
if 'bertscore_recall' in overall:
    print(f"  BERTScore Recall:   {overall.get('bertscore_recall', 0):.4f}")
print(f"  Throughput:          {overall.get('throughput', 0):.2f} images/sec")
print(f"  VRAM Usage:         {overall.get('vram_max_allocated_gb', 0):.2f} GB")

# Check if confusion matrices exist
cm = metrics.get('confusion_matrices', {})
if cm:
    print(f"\n✅ Confusion matrices generated:")
    if cm.get('binary_yes_no', {}).get('global'):
        print(f"  - Binary YES/NO (global)")
    if cm.get('state_open_closed', {}).get('global'):
        print(f"  - State OPEN/CLOSED (global)")
    if cm.get('state_up_down', {}).get('global'):
        print(f"  - State UP/DOWN (global)")
    if cm.get('navigation_elements', {}).get('global'):
        print(f"  - Navigation Elements (global)")

print("\n" + "=" * 60)
print("✅ Test completed successfully!")
print("=" * 60)